# EDA 009 Splitting 002: support-aware split validation

This notebook tests a **dual-track split** for both cold-start and history-enabled recommender evaluation.

## Proposed split policy (per user)

- **n = 1**: randomly assign the single row to `train` / `val` / `test` (cold-start users can appear in eval).
- **n = 2**: force `1` row in `train`, and `1` row in either `val` or `test` (random choice).
- **n >= 3**: force at least `1` row in `train`, and at least `2` rows in exactly one eval split (`val` or `test`, chosen randomly).

For `n >= 3`, any remaining rows are allocated between train and the chosen eval split with ratio-aware targets, while preserving the hard constraints above.

Design goals:

1. Preserve a true cold-start evaluation slice.
2. Keep history-enabled users with train support.
3. Ensure stronger eval signal for users with larger histories.
4. Avoid impossible cohort requests by construction.

In [8]:
from __future__ import annotations

from pathlib import Path
import time

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)


def _repo_root() -> Path:
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / "pyproject.toml").is_file():
            return d
    raise RuntimeError(f"Could not find repo root from cwd={here}")


def _uniform_from_user_ids(user_series: pd.Series, seed: int) -> np.ndarray:
    # Deterministic pseudo-uniform in [0,1) from user ids.
    as_str = user_series.astype("string").fillna("<NA>")
    h = pd.util.hash_pandas_object(as_str, index=False).to_numpy(dtype=np.uint64)
    salt = np.uint64(seed) * np.uint64(11400714819323198485)
    mixed = h ^ salt
    return mixed.astype(np.float64) / float(np.iinfo(np.uint64).max)


REPO_ROOT = _repo_root()
INPUT_PATH = REPO_ROOT / "data/interim/steam_reviews_cleaned_english.parquet"

USER_COL = "author.steamid"
TS_COL = "timestamp_created"
RID_COL = "review_id"
APP_COL = "app_id"
REC_COL = "recommended"

VAL_RATIO = 0.15
TEST_RATIO = 0.15
RNG_SEED = 2026
SAMPLE_USER_FRAC = 1.0  # set to 1.0 for full data
BATCH_SIZE = 250_000

if not INPUT_PATH.is_file():
    raise FileNotFoundError(INPUT_PATH)

usecols = [RID_COL, USER_COL, APP_COL, TS_COL, REC_COL]
pf = pq.ParquetFile(INPUT_PATH)

_t0 = time.perf_counter()
chunks = []
rows_full = 0
for batch in pf.iter_batches(batch_size=BATCH_SIZE, columns=usecols):
    chunk = batch.to_pandas()
    if len(chunk) == 0:
        continue

    rows_full += len(chunk)

    if SAMPLE_USER_FRAC < 1.0:
        u = _uniform_from_user_ids(chunk[USER_COL], seed=RNG_SEED)
        chunk = chunk.loc[u < SAMPLE_USER_FRAC]

    if len(chunk) == 0:
        continue

    chunks.append(chunk)

if chunks:
    df = pd.concat(chunks, ignore_index=True)
else:
    df = pd.DataFrame(columns=usecols)

df[TS_COL] = pd.to_numeric(df[TS_COL], errors="coerce")
_t1 = time.perf_counter()

print(f"Loaded rows (full scan): {rows_full:,}")
print(f"Loaded rows (working): {len(df):,}")
print(f"Unique users (working): {df[USER_COL].nunique():,}")
print(f"Load+sample wall time: {_t1 - _t0:,.2f}s")

Loaded rows (full scan): 9,160,492
Loaded rows (working): 9,160,492
Unique users (working): 5,108,608
Load+sample wall time: 0.60s


In [9]:
def build_support_aware_split_vectorized(
    df_all: pd.DataFrame,
    *,
    val_ratio: float,
    test_ratio: float,
    seed: int,
) -> pd.DataFrame:
    if len(df_all) == 0:
        out = df_all.copy()
        out["split"] = pd.Series(dtype="object")
        return out

    out = df_all.sort_values([USER_COL, TS_COL, RID_COL]).reset_index(drop=True).copy()

    n_interactions = out.groupby(USER_COL)[RID_COL].transform("size").astype(np.int64)
    pos_in_user = out.groupby(USER_COL).cumcount().astype(np.int64)

    # Deterministic user-level random choices from user hash + seed.
    user_u_cold = _uniform_from_user_ids(out[USER_COL], seed=seed + 11)
    user_u_eval = _uniform_from_user_ids(out[USER_COL], seed=seed + 29)

    eval_total = val_ratio + test_ratio
    p_val = 0.5 if eval_total <= 0 else (val_ratio / eval_total)
    eval_split_is_val = user_u_eval < p_val
    eval_split = np.where(eval_split_is_val, "val", "test")
    eval_ratio = np.where(eval_split_is_val, val_ratio, test_ratio)

    split = np.full(len(out), "train", dtype=object)

    # n = 1: random across train/val/test by global ratios.
    n1 = n_interactions == 1
    train_cut = max(0.0, 1.0 - val_ratio - test_ratio)
    split_n1 = np.where(
        user_u_cold < train_cut,
        "train",
        np.where(user_u_cold < train_cut + val_ratio, "val", "test"),
    )
    split[n1.to_numpy()] = split_n1[n1.to_numpy()]

    # n = 2: first row train, second row eval (val or test).
    n2 = n_interactions == 2
    n2_eval_row = n2 & (pos_in_user == 1)
    split[n2_eval_row.to_numpy()] = eval_split[n2_eval_row.to_numpy()]

    # n >= 3: >=1 train row and >=2 eval rows in a single eval dataset.
    n3p = n_interactions >= 3
    free_budget = (n_interactions - 3).clip(lower=0)
    extra_eval_target = np.rint(free_budget * eval_ratio).astype(np.int64)
    extra_eval = np.minimum(free_budget, np.maximum(0, extra_eval_target)).astype(np.int64)
    n_eval = (2 + extra_eval).astype(np.int64)
    n_eval = np.minimum(n_interactions - 1, n_eval)

    n3p_eval_row = n3p & (pos_in_user >= (n_interactions - n_eval))
    split[n3p_eval_row.to_numpy()] = eval_split[n3p_eval_row.to_numpy()]

    out["split"] = split
    return out


_t0 = time.perf_counter()
split_df = build_support_aware_split_vectorized(
    df,
    val_ratio=VAL_RATIO,
    test_ratio=TEST_RATIO,
    seed=RNG_SEED,
)
_t1 = time.perf_counter()

split_counts = split_df["split"].value_counts().reindex(["train", "val", "test"], fill_value=0)
print(f"Split assignment wall time: {_t1 - _t0:,.2f}s")
print("Rows train/val/test =", ", ".join(f"{k}:{v:,}" for k, v in split_counts.items()))

/tmp/ipykernel_76570/1915945801.py:26: RuntimeWarning: overflow encountered in scalar multiply
  salt = np.uint64(seed) * np.uint64(11400714819323198485)
/tmp/ipykernel_76570/1915945801.py:26: RuntimeWarning: overflow encountered in scalar multiply
  salt = np.uint64(seed) * np.uint64(11400714819323198485)


Split assignment wall time: 34.21s
Rows train/val/test = train:5,438,571, val:1,860,710, test:1,861,211


In [10]:
train = split_df.loc[split_df["split"] == "train"].copy()
val = split_df.loc[split_df["split"] == "val"].copy()
test = split_df.loc[split_df["split"] == "test"].copy()

ids_train, ids_val, ids_test = set(train[RID_COL]), set(val[RID_COL]), set(test[RID_COL])
overlap_tv = len(ids_train & ids_val)
overlap_tt = len(ids_train & ids_test)
overlap_vt = len(ids_val & ids_test)
print("Overlap review_id train∩val/train∩test/val∩test:", overlap_tv, overlap_tt, overlap_vt)

coverage = pd.DataFrame(
    {
        "split": ["train", "val", "test"],
        "rows": [len(train), len(val), len(test)],
        "unique_users": [train[USER_COL].nunique(), val[USER_COL].nunique(), test[USER_COL].nunique()],
        "unique_items": [train[APP_COL].nunique(), val[APP_COL].nunique(), test[APP_COL].nunique()],
    }
)
display(coverage)

train_users = set(train[USER_COL])
val_user_in_train = float(val[USER_COL].isin(train_users).mean()) if len(val) else np.nan
test_user_in_train = float(test[USER_COL].isin(train_users).mean()) if len(test) else np.nan
print(f"User-in-train coverage val/test: {val_user_in_train:.4f} / {test_user_in_train:.4f}")

user_counts = split_df.groupby(USER_COL)[RID_COL].count().rename("n_interactions")
split_df = split_df.merge(user_counts, left_on=USER_COL, right_index=True, how="left")
split_df["cohort"] = pd.cut(
    split_df["n_interactions"],
    bins=[0, 1, 2, 3, np.inf],
    labels=["n1", "n2", "n3", "n4_plus"],
)

cohort_summary = (
    split_df.groupby(["split", "cohort"], observed=True)
    .agg(
        rows=(RID_COL, "count"),
        unique_users=(USER_COL, "nunique"),
        recommended_rate=(REC_COL, "mean"),
    )
    .reset_index()
)
display(cohort_summary.sort_values(["split", "cohort"]))

# Constraint checks by user bucket
by_user = (
    split_df.groupby([USER_COL, "n_interactions", "split"], as_index=False)
    .agg(n_rows=(RID_COL, "count"))
)
wide = (
    by_user.pivot(index=[USER_COL, "n_interactions"], columns="split", values="n_rows")
    .fillna(0)
    .reset_index()
)
for c in ["train", "val", "test"]:
    if c not in wide.columns:
        wide[c] = 0

wide["n_eval"] = wide["val"] + wide["test"]
wide["eval_single_dataset"] = (wide["val"] == 0) | (wide["test"] == 0)

n2 = wide[wide["n_interactions"] == 2]
n3p = wide[wide["n_interactions"] >= 3]

n2_ok = ((n2["train"] == 1) & (n2["n_eval"] == 1)).mean() if len(n2) else np.nan
n3p_ok = ((n3p["train"] >= 1) & (n3p["n_eval"] >= 2) & (n3p["eval_single_dataset"])).mean() if len(n3p) else np.nan

print(f"Constraint pass rate n=2 (1 train + 1 eval): {n2_ok:.4f}")
print(f"Constraint pass rate n>=3 (>=1 train + >=2 eval in one split): {n3p_ok:.4f}")

bucket_audit = pd.DataFrame(
    {
        "bucket": ["n=1", "n=2", "n>=3"],
        "n_users": [
            int((wide["n_interactions"] == 1).sum()),
            int((wide["n_interactions"] == 2).sum()),
            int((wide["n_interactions"] >= 3).sum()),
        ],
        "avg_train_rows": [
            float(wide.loc[wide["n_interactions"] == 1, "train"].mean()) if (wide["n_interactions"] == 1).any() else np.nan,
            float(wide.loc[wide["n_interactions"] == 2, "train"].mean()) if (wide["n_interactions"] == 2).any() else np.nan,
            float(wide.loc[wide["n_interactions"] >= 3, "train"].mean()) if (wide["n_interactions"] >= 3).any() else np.nan,
        ],
        "avg_val_rows": [
            float(wide.loc[wide["n_interactions"] == 1, "val"].mean()) if (wide["n_interactions"] == 1).any() else np.nan,
            float(wide.loc[wide["n_interactions"] == 2, "val"].mean()) if (wide["n_interactions"] == 2).any() else np.nan,
            float(wide.loc[wide["n_interactions"] >= 3, "val"].mean()) if (wide["n_interactions"] >= 3).any() else np.nan,
        ],
        "avg_test_rows": [
            float(wide.loc[wide["n_interactions"] == 1, "test"].mean()) if (wide["n_interactions"] == 1).any() else np.nan,
            float(wide.loc[wide["n_interactions"] == 2, "test"].mean()) if (wide["n_interactions"] == 2).any() else np.nan,
            float(wide.loc[wide["n_interactions"] >= 3, "test"].mean()) if (wide["n_interactions"] >= 3).any() else np.nan,
        ],
    }
)
display(bucket_audit)

Overlap review_id train∩val/train∩test/val∩test: 0 0 0


,split,rows,unique_users,unique_items
0,train,5438571,4091753,315
1,val,1860710,1364932,315
2,test,1861211,1366682,315


User-in-train coverage val/test: 0.7277 / 0.7259


,split,cohort,rows,unique_users,recommended_rate
0,test,n1,510093,510093,0.897444
1,test,n2,445733,445733,0.899657
2,test,n3,359350,179675,0.891652
3,test,n4_plus,546035,231181,0.875644
4,train,n1,2376994,2376994,0.897197
5,train,n2,891744,891744,0.897226
6,train,n3,360594,360594,0.893484
7,train,n4_plus,1809239,462421,0.863351
8,val,n1,506762,506762,0.897924
9,val,n2,446011,446011,0.899213


Constraint pass rate n=2 (1 train + 1 eval): 1.0000
Constraint pass rate n>=3 (>=1 train + >=2 eval in one split): 1.0000


,bucket,n_users,avg_train_rows,avg_val_rows,avg_test_rows
0,n=1,3393849,0.700383,0.149318,0.150299
1,n=2,891744,1.000000,0.500156,0.499844
2,n>=3,823015,2.636444,1.103184,1.100083


In [11]:
# Recommender feasibility diagnostic: multi-positive val users and train-positive support

train_pos = (
    train.loc[train[REC_COL] == 1]
    .groupby(USER_COL)[RID_COL]
    .count()
    .rename("n_train_pos")
)

val_pos = val.loc[val[REC_COL] == 1].copy()
val_pos_user_stats = (
    val_pos.groupby(USER_COL)
    .agg(
        n_val_pos_rows=(RID_COL, "size"),
        n_val_pos_apps=(APP_COL, "nunique"),
    )
)

diag = val_pos_user_stats.join(train_pos, how="left").fillna({"n_train_pos": 0})
diag["n_train_pos"] = diag["n_train_pos"].astype(int)

diag["eval_pos_cohort"] = np.select(
    [diag["n_val_pos_apps"] >= 2, diag["n_val_pos_apps"] == 1],
    ["val_multi_pos_eval", "val_single_pos_eval"],
    default="val_no_pos_eval",
)

diag["train_pos_cohort"] = np.select(
    [diag["n_train_pos"] >= 2, diag["n_train_pos"] == 1, diag["n_train_pos"] == 0],
    ["train_multi_pos", "train_single_pos", "train_no_pos"],
    default="train_no_pos",
)

joint = (
    diag.groupby(["eval_pos_cohort", "train_pos_cohort"], as_index=False)
    .agg(n_users=("n_train_pos", "size"))
    .sort_values("n_users", ascending=False)
)
display(joint)

multi_eval = diag.loc[diag["eval_pos_cohort"] == "val_multi_pos_eval"]
with_train_pos = int((multi_eval["n_train_pos"] >= 1).sum())
all_multi_eval = int(len(multi_eval))
ratio = with_train_pos / all_multi_eval if all_multi_eval else float("nan")
print(f"Users with val_multi_pos_eval and >=1 train positive: {with_train_pos:,} / {all_multi_eval:,} ({ratio:.2%})")

,eval_pos_cohort,train_pos_cohort,n_users
4,val_single_pos_eval,train_no_pos,495736
5,val_single_pos_eval,train_single_pos,395557
0,val_multi_pos_eval,train_multi_pos,180409
2,val_multi_pos_eval,train_single_pos,148293
3,val_single_pos_eval,train_multi_pos,23714
1,val_multi_pos_eval,train_no_pos,12464


Users with val_multi_pos_eval and >=1 train positive: 328,702 / 341,166 (96.35%)


In [12]:
# Assuming train dataset is stored as a CSV or parquet file, 
# but since in this notebook 'train' is already used above, 
# likely as a DataFrame resulting from a prior split/load step.
# For explicit re-reading (if needed):

# If CSV:
# train = pd.read_csv('path/to/train.csv')

# If parquet:
train = pd.read_parquet(INPUT_PATH)

In [13]:
# Count number of users (author.steamid) with 1, 2, 3, or >=4 records in the train DataFrame
user_counts = train['author.steamid'].value_counts()
n_1 = (user_counts == 1).sum()
n_2 = (user_counts == 2).sum()
n_3 = (user_counts == 3).sum()
n_4plus = (user_counts >= 4).sum()

print(f"Users with 1 record: {n_1}")
print(f"Users with 2 records: {n_2}")
print(f"Users with 3 records: {n_3}")
print(f"Users with 4 or more records: {n_4plus}")

Users with 1 record: 3393849
Users with 2 records: 891744
Users with 3 records: 360594
Users with 4 or more records: 462421


## Task A preflight feasibility gate

This section validates whether the split is likely to produce enough Task A signal **before** running `recs_004_eval_proxy_same_user_task_a.ipynb`.

Checks:

- `val_user_in_train` (val rows whose user appears in train)
- `users_with_>=3_val_pos_apps` (users with at least 3 unique positive val apps)
- `proxy_coverage_multi_pos` (share of val-positive examples with `n_pos >= 2` under query-holdout logic)

If any required threshold fails, this cell raises `RuntimeError`.

In [14]:
# Task A feasibility thresholds (tune as needed)
MIN_VAL_USER_IN_TRAIN = 0.70
MIN_USERS_GE3_VAL_POS_APPS = 0.05
MIN_PROXY_COVERAGE_MULTI_POS = 0.08

# Expected columns from prior cells: train, val, USER_COL, APP_COL, REC_COL
required_vars = ["train", "val", "USER_COL", "APP_COL", "REC_COL"]
missing_vars = [v for v in required_vars if v not in globals()]
if missing_vars:
    raise RuntimeError(f"Missing required objects from prior cells: {missing_vars}")

train_users = set(train[USER_COL])
val_user_in_train = float(val[USER_COL].isin(train_users).mean()) if len(val) else np.nan

val_pos = val.loc[val[REC_COL] == 1].copy()
if len(val_pos) == 0:
    raise RuntimeError("No positive val rows found; Task A cannot be evaluated.")

# User-level val positive app counts
val_pos_user_apps = (
    val_pos.groupby(USER_COL)[APP_COL]
    .nunique()
    .rename("n_val_pos_apps")
)

share_users_ge3_val_pos = float((val_pos_user_apps >= 3).mean()) if len(val_pos_user_apps) else np.nan

# Example-level proxy: each positive val row is a potential query; n_pos = other val-positive apps
val_pos = val_pos.merge(val_pos_user_apps, left_on=USER_COL, right_index=True, how="left")
val_pos["n_pos_proxy"] = (val_pos["n_val_pos_apps"] - 1).clip(lower=0)
proxy_coverage_multi_pos = float((val_pos["n_pos_proxy"] >= 2).mean()) if len(val_pos) else np.nan

preflight = pd.DataFrame(
    [
        {
            "metric": "val_user_in_train",
            "value": val_user_in_train,
            "threshold": MIN_VAL_USER_IN_TRAIN,
            "pass": bool(val_user_in_train >= MIN_VAL_USER_IN_TRAIN),
        },
        {
            "metric": "share_users_ge3_val_pos_apps",
            "value": share_users_ge3_val_pos,
            "threshold": MIN_USERS_GE3_VAL_POS_APPS,
            "pass": bool(share_users_ge3_val_pos >= MIN_USERS_GE3_VAL_POS_APPS),
        },
        {
            "metric": "proxy_coverage_multi_pos",
            "value": proxy_coverage_multi_pos,
            "threshold": MIN_PROXY_COVERAGE_MULTI_POS,
            "pass": bool(proxy_coverage_multi_pos >= MIN_PROXY_COVERAGE_MULTI_POS),
        },
    ]
)

print("Task A preflight feasibility report:")
display(preflight)

failed = preflight.loc[~preflight["pass"], ["metric", "value", "threshold"]]
if len(failed) > 0:
    fail_msg = " | ".join(
        f"{r.metric}: value={r.value:.4f} < threshold={r.threshold:.4f}"
        for r in failed.itertuples(index=False)
    )
    raise RuntimeError(
        "Task A preflight failed. Improve split support before running recs_004. " + fail_msg
    )

print("Task A preflight passed. Split support looks sufficient for recs_004 Task A.")

Task A preflight feasibility report:


,metric,value,threshold,pass
0,val_user_in_train,1.000000,0.70,True
1,share_users_ge3_val_pos_apps,0.038476,0.05,False
2,proxy_coverage_multi_pos,0.094487,0.08,True


RuntimeError: Task A preflight failed. Improve split support before running recs_004. share_users_ge3_val_pos_apps: value=0.0385 < threshold=0.0500